In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# !pip install datasets huggingface_hub 
# !pip install git+https://github.com/huggingface/transformers
# !pip install transformers==4.27.0
# !pip install evaluate
# !pip install lime

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from transformers import BertTokenizer, BertForQuestionAnswering, BertConfig

import torch
from torch.utils.data.dataloader import DataLoader

from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, EvalPrediction, GlueDataset
from transformers import AutoTokenizer, AutoModel, AutoModelWithLMHead
from transformers import GlueDataTrainingArguments as DataTrainingArguments


import lime
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2"
torch.cuda.empty_cache()

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

In [5]:
torch.cuda.device_count()

3

In [6]:
# for i in range(3):
#     print(torch.cuda.get_device_name(i))
#     print('Memory Usage:')
#     print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
#     print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

The first step is to fine-tune BERT model on SQUAD dataset. This can be easiy accomplished by following the steps described in hugging face's official web site: https://github.com/huggingface/transformers#run_squadpy-fine-tuning-on-squad-for-question-answering 

Note that the fine-tuning is done on a `bert-base-uncased` pre-trained model.

After we pretrain the model, we can load the tokenizer and pre-trained BERT model using the commands described below. 

In [7]:
# replace <PATH-TO-SAVED-MODEL> with the real path of the saved model
model_path = 'randellcotta/distilbert-base-uncased-finetuned-yelp-polarity'

# load model
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# parallelising
# model=nn.DataParallel(model)
model.to(device)
# model.eval()
# model.zero_grad()

# load tokenizer
tokenizer=AutoTokenizer.from_pretrained(model_path)

# Data loading and tokenization

In [8]:
question_neg, text_neg = "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually takes 2 hours of repeated calling to get an answer. Who has time for that or wants to deal with it? I have run into this problem with many other doctors and I just don't get it. You have office workers, you have patients with medical needs, why isn't anyone answering the phone? It's incomprehensible and not work the aggravation. It's with regret that I feel that I have to give Dr. Goldberg 2 stars.", '0'
question_pos, text_pos = "Before I finally made it over to this range I heard the same thing from most people - it's just fine to go work on your swing. I had such a low expectation I was pleasantly surprised. \n\nIt's a fairly big range - if you are familiar with Scally's in Moon, it seems like it has almost as many tees, though its not nearly as nice a facility. \n\nThe guys in the pro shop were two of the friendlier guys I've come across at ranges or at courses. Yards were indeed marked and there are some targets to aim for, and even some hazards to aim away from. \n\nA big red flag to me was the extra charge ($3) to hit off the grass. I am no range expert, but this is the 4th one I've been to and the first I've seen of that sort of nickel and diming....\n\nPrice for the golf balls was reasonable and I do plan to be back every week until they close up in October for the season. Hopefully, since its for sale, it will reopen as a golf facility again.",'1'

In [9]:
tokenized_stuff_neg=tokenizer.encode(question_neg,add_special_tokens=False,return_tensors='pt').to(device)
tokenized_stuff_pos=tokenizer.encode(question_pos,add_special_tokens=False,return_tensors='pt').to(device)

In [10]:
with torch.no_grad():
    out_neg = model(input_ids=tokenized_stuff_neg.to(device))
    out_pos = model(input_ids=tokenized_stuff_pos.to(device))

In [11]:
print("Negative Sample Prediction ", out_neg)
print("Positive Sample Prediction ", out_pos)

Negative Sample Prediction  SequenceClassifierOutput(loss=None, logits=tensor([[ 3.5445, -3.7237]], device='cuda:0'), hidden_states=None, attentions=None)
Positive Sample Prediction  SequenceClassifierOutput(loss=None, logits=tensor([[-1.7218,  1.5928]], device='cuda:0'), hidden_states=None, attentions=None)


# Running with the Yelp + SST data

In [12]:
%cd Spurious_Correlations-20230425T172048Z-001/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models/
# %cd /content/drive/MyDrive/CMSC848D_Project/Spurious_Correlations-20230425T172048Z-001/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models

/home/harduin/Desktop/Research/CMSC848D_Project/Spurious_Correlations-20230425T172048Z-001/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models


In [13]:
import utils.data_structure as dataproj

In [14]:
df_yelp=dataproj.get_yelp()
df_sst2=dataproj.get_sst2()

Found cached dataset yelp_polarity (/home/harduin/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61)


  0%|          | 0/2 [00:00<?, ?it/s]

Found cached dataset glue (/home/harduin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


  0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
# from lime.lime_text import LimeTextExplainer
# explainer = LimeTextExplainer(class_names=['Negative','Positive'])

In [16]:
df_yelp['text'][0]

"Contrary to other reviews, I have zero complaints about the service or the prices. I have been getting tire service here for the past 5 years now, and compared to my experience with places like Pep Boys, these guys are experienced and know what they're doing. \\nAlso, this is one place that I do not feel like I am being taken advantage of, just because of my gender. Other auto mechanics have been notorious for capitalizing on my ignorance of cars, and have sucked my bank account dry. But here, my service and road coverage has all been well explained - and let up to me to decide. \\nAnd they just renovated the waiting room. It looks a lot better than it did in previous years."

In [17]:
# exp = explainer.explain_instance(df_yelp['text'][0],fun_predict, num_samples=1000,num_features=100)

In [18]:
# import time

# word2id=dict()
# id2word=dict()

# idx_sentence=0

# for idx in range(df_yelp.shape[0]):
#     start=time.time()
#     exp = explainer.explain_instance(df_yelp['text'][idx],fun_predict, num_samples=500,num_features=100)
#     out=exp.as_list()
# #     for i in out:
# #         id2word[idx]=i[0]
# #         print(id2word)
# #     print("ONE INDEX")
        
# # print(id2word)
#     for i in out:
#         id2word[idx]=i[0]
# #         print("ID2WORD \n \n",id2word)
# #         print("Word : ", i[0])
# #         print("Value : ", i[1])
#         if (i[0] in word2id.keys()):
# #             print("APPEND")
# #             print("##################### DEBUG ###################")
# #             print("word2id[i[0]][0] ", word2id[i[0]][0])
# #             print("word2id[i[0]][1] ",word2id[i[0]][1])
# #             print("##################### DEBUG ###################")
#             word2id[i[0]][0].append(i[1])
#             word2id[i[0]][1].append(idx)
#         else:
# #             print("CREATE")
#             word2id[i[0]]=[[],[]]
#             word2id[i[0]][0].append(i[1])
#             word2id[i[0]][1].append(idx)
#     end=time.time()
#     print("Index : ",idx, " Time elapsed : ",end-start)
# #         print("Word2ID \n \n",word2id)
# #         print(word2id)
# #     print("One index finished \n \n \n")

# print(id2word)
# print(word2id)
# # print(len(list(word2id.keys())))



# # print(len(out))
    
# # word2id.keys()



# # x=pd.DataFrame(columns=['word','value'])
# # for i in range(df_yelp.shape[0]):
# #     exp = explainer.explain_instance(df_yelp['text'][i],fun_predict, num_samples=1000,num_features=100)
# #     print(exp.as_list())
# #     print('\n')
# #     print('\n')
# #     print('\n')
# #     df=pd.DataFrame(exp.as_list(),columns=['train_idx','word','value'])
# # #     print(count)
# #     if i==5:
# #         break
# # # count

In [21]:
# word2id

In [22]:
# import pickle
# with open('saved_word2id.pkl','wb') as f:
#     pickle.dump(word2id,f)
# with open('saved_id2word.pkl','wb') as g:
#     pickle.dump(id2word,g)

In [23]:
# len(set(word2id.keys()))

In [24]:
# with open('saved_word2id.pkl', 'rb') as f:
#     x = pickle.load(f)
# print(x)

In [25]:
# import time

# word2id=dict()
# id2word=dict()

# idx_sentence=0

# for idx in range(df_yelp.shape[0]):
#     start=time.time()
#     exp = explainer.explain_instance(df_yelp['text'][idx],fun_predict, num_samples=250,num_features=51)
#     out=exp.as_list()
# #     for i in out:
# #         id2word[idx]=i[0]
# #         print(id2word)
# #     print("ONE INDEX")
        
# # print(id2word)
#     for i in out:
#         id2word[idx]=i[0]
# #         print("ID2WORD \n \n",id2word)
# #         print("Word : ", i[0])
# #         print("Value : ", i[1])
#         if (i[0] in word2id.keys()):
# #             print("APPEND")
# #             print("##################### DEBUG ###################")
# #             print("word2id[i[0]][0] ", word2id[i[0]][0])
# #             print("word2id[i[0]][1] ",word2id[i[0]][1])
# #             print("##################### DEBUG ###################")
#             word2id[i[0]][0].append(i[1])
#             word2id[i[0]][1].append(idx)
#         else:
# #             print("CREATE")
#             word2id[i[0]]=[[],[]]
#             word2id[i[0]][0].append(i[1])
#             word2id[i[0]][1].append(idx)
#     end=time.time()
#     print("Index : ",idx, " Time elapsed : ",end-start)
# #         print("Word2ID \n \n",word2id)
# #         print(word2id)
# #     print("One index finished \n \n \n")

# print(id2word)
# print(word2id)


# Captum implementation LIG

In [26]:
# !pip install captum

In [28]:
# fun_predict_captum(tokenized_stuff_neg)

Negative

In [19]:
from captum.attr import LayerGradientXActivation as LGA
# from captum._utils.models.linear_model import SkLearnLasso

def fun_predict_captum(tokenized_stuff_neg):
  # print("Yo")
  out_neg = model(tokenized_stuff_neg.to(torch.long))
  # print(out_neg)
  out_neg=out_neg[0].to(device)
  # print(out_neg)
  # print(device)
  # classfn_out_neg=torch.softmax(out_neg,dim=1).detach().cpu().numpy()
  classfn_out_neg=torch.softmax(out_neg,dim=1)
  # print(classfn_out_neg[0])
  return classfn_out_neg

lga = LGA(fun_predict_captum,model.distilbert.embeddings)
attr = lga.attribute(tokenized_stuff_neg, target=0)

In [20]:
from captum.attr import visualization as viz

indices = tokenized_stuff_neg[0].detach().tolist()
all_tokens = tokenizer.convert_ids_to_tokens(indices)
confidence=torch.max(fun_predict_captum(tokenized_stuff_neg)[0])
label_pred=torch.argmax(fun_predict_captum(tokenized_stuff_neg)[0])
ground_label=0
# print(end_scores)
delta=torch.Tensor([0])

def summarize_attributions(attributions):
    attributions = attributions.sum(dim=-1).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    # print(attributions)
    return attributions

attributions_start_sum = summarize_attributions(attr)

#  Change first 3 zero to 1 if positive class

start_position_vis = viz.VisualizationDataRecord(
                        attributions_start_sum,
                        confidence, 
                        label_pred,
                        ground_label,
                        0,
                        attributions_start_sum.sum(),       
                        all_tokens,
                        delta)

print('\033[1m', 'Visualizations For Start Position', '\033[0m')
viz.visualize_text([start_position_vis])

 Visualizations For Start Position 


In [21]:
attr_coag=summarize_attributions(attr)

In [22]:
from IPython.core.display import HTML, display
# import nltk
# nltk.download('punkt')
def show_text_attr(attrs):
    rgb = lambda x: '255,0,0' if x < 0 else '0,255,0'
    alpha = lambda x: abs(x) ** 0.5
    # for token, attr in zip(tokenizer(question_pos), attrs.tolist()):
    #   print(attr) 
    token_marks = [
        f'<mark style="background-color:rgba({rgb(attr)},{alpha(attr)})">{token}</mark>'
        for token, attr in zip(tokenizer.convert_ids_to_tokens(tokenized_stuff_neg[0].detach().tolist()), attrs.tolist())
    ]
    # print(token_marks)
    display(HTML('<p>' + ' '.join(token_marks) + '</p>'))
    
show_text_attr(attr_coag)

<ipython-input-22-59bfd5c23f0c>:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import HTML, display


Positive

In [23]:
lga.has_convergence_delta()

False

In [24]:
from captum.attr import LayerGradientXActivation as LGA
# from captum._utils.models.linear_model import SkLearnLasso

def fun_predict_captum(tokenized_stuff_pos):
  # print("Yo")
  out_neg = model(tokenized_stuff_pos.to(torch.long))
  # print(out_neg)
  out_neg=out_neg[0].to(device)
  # print(out_neg)
  # print(device)
  # classfn_out_neg=torch.softmax(out_neg,dim=1).detach().cpu().numpy()
  classfn_out_neg=torch.softmax(out_neg,dim=1)
  return classfn_out_neg

lga = LGA(fun_predict_captum,model.distilbert.embeddings)
attr = lga.attribute(tokenized_stuff_pos, target=1)

In [25]:
delta[0]=0
delta

tensor([0.])

In [26]:
from captum.attr import visualization as viz

indices = tokenized_stuff_pos[0].detach().tolist()
all_tokens = tokenizer.convert_ids_to_tokens(indices)

def summarize_attributions(attributions):
    attributions = attributions.sum(dim=-1).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    # print(attributions)
    return attributions

attributions_start_sum = summarize_attributions(attr)

#  Change first 3 zero to 1 if positive class

start_position_vis = viz.VisualizationDataRecord(
                        attributions_start_sum,
                        1, 
                        1,
                        1,
                        1,
                        attributions_start_sum.sum(),       
                        all_tokens,
                        delta)

print('\033[1m', 'Visualizations For Start Position', '\033[0m')
viz.visualize_text([start_position_vis])

 Visualizations For Start Position 


In [38]:
attr_coag=summarize_attributions(attr)

In [39]:
from IPython.core.display import HTML, display
# import nltk
# nltk.download('punkt')
def show_text_attr(attrs):
    rgb = lambda x: '255,0,0' if x < 0 else '0,255,0'
    alpha = lambda x: abs(x) ** 0.5
    # for token, attr in zip(tokenizer(question_pos), attrs.tolist()):
    #   print(attr) 
    token_marks = [
        f'<mark style="background-color:rgba({rgb(attr)},{alpha(attr)})">{token}</mark>'
        for token, attr in zip(tokenizer.convert_ids_to_tokens(tokenized_stuff_pos[0].detach().tolist()), attrs.tolist())
    ]
    # print(token_marks)
    display(HTML('<p>' + ' '.join(token_marks) + '</p>'))
    
show_text_attr(attr_coag)

<ipython-input-39-4d7d0ca19df6>:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import HTML, display


In [28]:
# Define the sample from the dataset

import utils.data_structure as ds

df_yelp=ds.get_yelp()
df_sst=ds.get_sst2()
df_movie_rationales=ds.get_movie_rationales()

sentences_yelp=df_yelp['text']
labels_yelp=df_yelp['label']

sentences_sst=df_sst['text']
labels_sst=df_sst['label']

sentences_mr=df_movie_rationales['text']
labels_mr=df_movie_rationales['label']

idx=2017

sentence=sentences_yelp[idx]
label=labels_yelp[idx]

tokenized_sentence=tokenizer.encode(sentence,add_special_tokens=False,padding=True,truncation=True,return_tensors='pt').to(device)

# Compute LIME attribution

from captum.attr import Lime
from captum._utils.models.linear_model import SkLearnLasso

def fun_predict_captum(tokenized_stuff_pos):
  out_neg = model(tokenized_stuff_pos)
  out_neg=out_neg[0].to(device)
  classfn_out_neg=torch.softmax(out_neg,dim=1)
  return classfn_out_neg

lga = LGA(fun_predict_captum,model.distilbert.embeddings)
attr = lga.attribute(tokenized_stuff_pos, target=0)
        
def summarize_attributions(attributions):
    attributions = attributions.sum(dim=-1).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    # print(attributions)
    return attributions

temp=summarize_attributions(attr)

for i in range(len(temp)):
    if temp[i]>0 or temp[i]<0:
        temp[i]=temp[i]*100

# Visualize the attribution for the attribution

from captum.attr import visualization as viz

indices = tokenized_sentence[0].detach().tolist()
all_tokens = tokenizer.convert_ids_to_tokens(indices)
delta=torch.Tensor([0])
confidence=torch.max(fun_predict_captum(tokenized_sentence)[0])
label_pred=torch.argmax(fun_predict_captum(tokenized_sentence)[0])
ground_label=0
last_arg=0

attributions_start_sum = attr[0]

#  Change first 3 zero to 1 if positive class

start_position_vis = viz.VisualizationDataRecord(
                        temp,
                        confidence, 
                        label_pred,
                        ground_label,
                        last_arg,
                        -1*temp.sum(),       
                        all_tokens,
                        delta)

print('\033[1m', 'Visualizations For Start Position', '\033[0m')
viz.visualize_text([start_position_vis])



Found cached dataset yelp_polarity (/home/harduin/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61)


  0%|          | 0/2 [00:00<?, ?it/s]

Found cached dataset glue (/home/harduin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


  0%|          | 0/3 [00:00<?, ?it/s]

Found cached dataset movie_rationales (/home/harduin/.cache/huggingface/datasets/movie_rationales/default/0.1.0/70ed6b72496c90835e8ee73ebf8d0e49f5ad3aa93f302c8a4b6c886143cfb779)


  0%|          | 0/3 [00:00<?, ?it/s]

 Visualizations For Start Position 
